In [ ]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist["data"]
y = mnist["target"].astype(np.int64)

print(X.shape, y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=5000, train_size=30000, random_state=42, stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_acc)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

knn_pred = knn.predict(X_test)
knn_acc = accuracy_score(y_test, knn_pred)

print("KNN Accuracy:", knn_acc)


In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

sgd = make_pipeline(
    StandardScaler(),
    SGDClassifier(random_state=42)
)

sgd.fit(X_train, y_train)

sgd_pred = sgd.predict(X_test)
sgd_acc = accuracy_score(y_test, sgd_pred)

print("SGD Accuracy:", sgd_acc)


In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)

nb_pred = nb.predict(X_test)
nb_acc = accuracy_score(y_test, nb_pred)

print("Naive Bayes Accuracy:", nb_acc)


In [ ]:
from sklearn.ensemble import VotingClassifier

voting_hard = VotingClassifier(
    estimators=[
        ("rf", rf),
        ("knn", knn),
        ("sgd", sgd),
        ("nb", nb)
    ],
    voting="hard"
)

voting_hard.fit(X_train, y_train)

hard_pred = voting_hard.predict(X_test)
hard_acc = accuracy_score(y_test, hard_pred)

print("Hard Voting Ensemble Accuracy:", hard_acc)


In [ ]:
sgd_soft = make_pipeline(
    StandardScaler(),
    SGDClassifier(loss="log_loss", random_state=42)
)

sgd_soft.fit(X_train, y_train)


In [ ]:
voting_soft = VotingClassifier(
    estimators=[
        ("rf", rf),
        ("knn", knn),
        ("sgd", sgd_soft),
        ("nb", nb)
    ],
    voting="soft"
)

voting_soft.fit(X_train, y_train)

soft_pred = voting_soft.predict(X_test)
soft_acc = accuracy_score(y_test, soft_pred)

print("Soft Voting Ensemble Accuracy:", soft_acc)


In [ ]:
results = {
    "Random Forest": rf_acc,
    "KNN": knn_acc,
    "SGD": sgd_acc,
    "Naive Bayes": nb_acc,
    "Hard Voting Ensemble": hard_acc
}

# Add soft voting only if you ran it
# results["Soft Voting Ensemble"] = soft_acc

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")


In this activity, I learned how ensemble learning can improve model performance by combining multiple classifiers instead of relying on a single one. Each individual model has different strengths and weaknesses. For example, KNN performs well on MNIST because it compares digit similarity, while SGD is fast and works well with large datasets but may not always reach the highest accuracy. Random Forest is strong and stable, and Naive Bayes is simple but can struggle on image-based datasets.

After combining the models using a VotingClassifier, the ensemble model performed better than some individual classifiers and gave more stable performance overall. The ensemble worked best when the individual models made different kinds of mistakes, because voting helped correct errors made by one model using the predictions of the others.

One interesting observation was that the ensemble did not always outperform the best individual model. If one model (like Random Forest or KNN) already performs very well, the ensemble may only slightly improve accuracy or sometimes stay similar. Overall, I learned that ensembles are most beneficial when the models are diverse, and they help improve reliability and reduce overfitting.